In [1]:
#!/usr/bin/env python3
# =========================================================
# FACTORIZATION MACHINE FOR YELP DATASET
# CuPy Implementation - KAGGLE VERSION
# Improvements:
#   [1] City + Category as separate embedding entities
#   [2] k=128
#   [3] 200 epochs, full 7M data
#   [4] Chi-square selected features (15)
# =========================================================

import json
import cupy as cp
import numpy as np
import math
from datetime import datetime
import time


# =========================================================
# ID MAPPER
# =========================================================
class IDMapper:
    def __init__(self):
        self.map = {}

    def get(self, key):
        if key not in self.map:
            self.map[key] = len(self.map)
        return self.map[key]

    def size(self):
        return len(self.map)


# =========================================================
# LOAD CONTEXTS
# =========================================================
def load_user_context(path):
    print(f"Loading user context from {path}...")
    ctx = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            u   = json.loads(line)
            uid = u["user_id"]
            rc  = max(0, u.get("review_count", 0) or 0)
            ctx[uid] = {
                "review_count":  rc,
                "average_stars": max(0, u.get("average_stars", 0) or 0),
                "fans":          max(0, u.get("fans", 0) or 0),
                "user_level":    (0 if rc < 10 else 1 if rc < 50 else 2 if rc < 200 else 3),
            }
    print(f"  ✓ Loaded {len(ctx):,} users")
    return ctx


def load_business_context(path):
    print(f"Loading business context from {path}...")
    ctx      = {}
    city_map = IDMapper()
    cat_map  = IDMapper()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            b        = json.loads(line)
            bid      = b["business_id"]
            cats     = b.get("categories") or ""
            main_cat = cats.split(",")[0].strip() if cats else "Unknown"
            city     = (b.get("city") or "Unknown").strip()

            ctx[bid] = {
                "stars":        max(0, b.get("stars", 0) or 0),
                "review_count": max(0, b.get("review_count", 0) or 0),
                "city_idx":     city_map.get(city),
                "cat_idx":      cat_map.get(main_cat),
            }

    print(f"  ✓ Loaded {len(ctx):,} businesses")
    print(f"  ✓ Unique cities: {city_map.size():,}, categories: {cat_map.size():,}")
    return ctx, city_map.size(), cat_map.size()


def load_checkin_context(path):
    print(f"Loading checkin context from {path}...")
    ctx = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            c     = json.loads(line)
            bid   = c["business_id"]
            dates = c.get("date", "")
            ctx[bid] = len(dates.split(", ")) if dates else 0
    print(f"  ✓ Loaded {len(ctx):,} businesses with checkin data")
    return ctx


def load_tip_context(path):
    print(f"Loading tip context from {path}...")
    user_tip = {}
    biz_tip  = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            t      = json.loads(line)
            uid    = t["user_id"]
            bid    = t["business_id"]
            length = len(t.get("text", ""))

            if uid not in user_tip:
                user_tip[uid] = [0, 0]  # count, total_length
            user_tip[uid][0] += 1
            user_tip[uid][1] += length

            if bid not in biz_tip:
                biz_tip[bid] = [0]      # count only
            biz_tip[bid][0] += 1

    print(f"  ✓ Loaded tips for {len(user_tip):,} users and {len(biz_tip):,} businesses")
    return user_tip, biz_tip


# =========================================================
# DATA LOADER
# =========================================================
# 13 numerical features (bỏ city_idx, cat_idx khỏi num — thay bằng embedding)
# + city_indices, cat_indices riêng
#
# Numerical (13):
#  f01 user_avg_stars       f02 biz_stars
#  f03 user_review_count    f04 user_level
#  f05 user_fans            f06 review_text_length
#  f07 biz_checkin_count    f08 biz_tip_count
#  f09 review_year          f10 biz_review_count
#  f11 user_tip_count       f12 review_cool
#  f13 user_tip_avg_length
#
# Embedding entities: user, business, city, category

def load_reviews(review_path, user_ctx, biz_ctx, checkin_ctx, user_tip, biz_tip,
                 user_map, biz_map, max_reviews=None):
    print(f"\nLoading reviews from {review_path}...")
    if max_reviews:
        print(f"  ⚠️  Limited to first {max_reviews:,} reviews")

    user_indices = []
    biz_indices  = []
    city_indices = []
    cat_indices  = []
    num_features = []
    ratings      = []
    count        = 0

    with open(review_path, "r", encoding="utf-8") as f:
        for line in f:
            if max_reviews and count >= max_reviews:
                break

            r   = json.loads(line)
            uid = r["user_id"]
            bid = r["business_id"]

            uidx = user_map.get(uid)
            bidx = biz_map.get(bid)

            u  = user_ctx.get(uid, {})
            b  = biz_ctx.get(bid, {})
            ut = user_tip.get(uid, [0, 0])
            bt = biz_tip.get(bid, [0])
            dt = datetime.strptime(r["date"], "%Y-%m-%d %H:%M:%S")

            # ===== 13 NUMERICAL FEATURES =====
            f01 = u.get("average_stars", 0) / 5.0
            f02 = b.get("stars", 0) / 5.0
            f03 = math.log1p(u.get("review_count", 0)) / 10.0
            f04 = u.get("user_level", 0) / 3.0
            f05 = math.log1p(u.get("fans", 0)) / 10.0
            f06 = math.log1p(len(r.get("text", "") or "")) / 10.0
            f07 = math.log1p(checkin_ctx.get(bid, 0)) / 10.0
            f08 = math.log1p(bt[0]) / 10.0
            f09 = (dt.year - 2005) / 20.0
            f10 = math.log1p(b.get("review_count", 0)) / 10.0
            f11 = math.log1p(ut[0]) / 10.0
            f12 = math.log1p(min(max(int(r.get("cool", 0) or 0), 0), 100))
            f13 = math.log1p(max(ut[1] / max(ut[0], 1), 0)) / 10.0

            num_feat = np.array(
                [f01, f02, f03, f04, f05, f06, f07,
                 f08, f09, f10, f11, f12, f13],
                dtype=np.float32
            )

            user_indices.append(uidx)
            biz_indices.append(bidx)
            city_indices.append(b.get("city_idx", 0))
            cat_indices.append(b.get("cat_idx", 0))
            num_features.append(num_feat)
            ratings.append(float(r["stars"]))

            count += 1
            if count % 100000 == 0:
                print(f"  Loaded {count:,} reviews...")

    print(f"  ✓ Total: {count:,} reviews loaded")
    print(f"  ✓ Unique users: {user_map.size():,}")
    print(f"  ✓ Unique businesses: {biz_map.size():,}")
    print(f"  ✓ Numerical features per sample: {len(num_features[0])}")

    return (
        np.array(user_indices,  dtype=np.int32),
        np.array(biz_indices,   dtype=np.int32),
        np.array(city_indices,  dtype=np.int32),
        np.array(cat_indices,   dtype=np.int32),
        np.array(num_features,  dtype=np.float32),
        np.array(ratings,       dtype=np.float32)
    )


# =========================================================
# FACTORIZATION MACHINE — WITH CITY & CATEGORY EMBEDDINGS
# =========================================================
class FactorizationMachine:
    """
    FM với 4 embedding entities: user, business, city, category
    + 13 numerical features
    City và category được học embedding riêng thay vì normalize index
    """

    def __init__(self, n_users, n_businesses, n_cities, n_cats, num_dim,
                 k=128, learning_rate=0.01, reg=0.001,
                 dropout_rate=0.3, grad_clip=5.0):

        self.n_users      = n_users
        self.n_businesses = n_businesses
        self.n_cities     = n_cities
        self.n_cats       = n_cats
        self.num_dim      = num_dim
        self.k            = k
        self.lr           = learning_rate
        self.reg          = reg
        self.dropout_rate = dropout_rate
        self.grad_clip    = grad_clip
        self.training     = True

        print(f"\n🔧 Initializing FM model on GPU:")
        print(f"  Users: {n_users:,} | Businesses: {n_businesses:,}")
        print(f"  Cities: {n_cities:,} | Categories: {n_cats:,}")
        print(f"  Numerical features: {num_dim}")
        print(f"  Embedding dimension: {k}")
        print(f"  Reg: {reg} | Dropout: {dropout_rate}")

        scale = float(1.0 / math.sqrt(k))

        # Global bias
        self.bias = cp.float32(0.0)

        # Linear weights
        self.user_lin = cp.zeros(n_users,      dtype=cp.float32)
        self.biz_lin  = cp.zeros(n_businesses, dtype=cp.float32)
        self.city_lin = cp.zeros(n_cities,     dtype=cp.float32)
        self.cat_lin  = cp.zeros(n_cats,       dtype=cp.float32)
        self.num_lin  = cp.zeros(num_dim,      dtype=cp.float32)

        # Embeddings
        self.user_emb = cp.random.randn(n_users,      k).astype(cp.float32) * scale
        self.biz_emb  = cp.random.randn(n_businesses, k).astype(cp.float32) * scale
        self.city_emb = cp.random.randn(n_cities,     k).astype(cp.float32) * scale
        self.cat_emb  = cp.random.randn(n_cats,       k).astype(cp.float32) * scale
        self.num_emb  = cp.random.randn(num_dim,       k).astype(cp.float32) * scale

        # Tính RAM ước tính
        total_params = (
            n_users * k + n_businesses * k +
            n_cities * k + n_cats * k + num_dim * k
        )
        print(f"  Total embedding params: {total_params:,} ({total_params*4/1e6:.1f} MB)")
        print("  ✓ Model initialized on GPU")

    def _clip(self, grad):
        norm = cp.linalg.norm(grad)
        if norm > self.grad_clip:
            grad = grad * (self.grad_clip / (norm + 1e-8))
        return grad

    def predict_batch(self, user_idx, biz_idx, city_idx, cat_idx, num_feat):
        batch_size = len(user_idx)

        # Linear part
        linear  = cp.full(batch_size, self.bias, dtype=cp.float32)
        linear += self.user_lin[user_idx]
        linear += self.biz_lin[biz_idx]
        linear += self.city_lin[city_idx]
        linear += self.cat_lin[cat_idx]
        linear += cp.sum(self.num_lin * num_feat, axis=1)

        # Embedding interaction (5 entities)
        u_emb = self.user_emb[user_idx]               # (B, k)
        b_emb = self.biz_emb[biz_idx]                 # (B, k)
        c_emb = self.city_emb[city_idx]               # (B, k)
        a_emb = self.cat_emb[cat_idx]                 # (B, k)
        n_emb = cp.dot(num_feat, self.num_emb)        # (B, k)

        stacked     = cp.stack([u_emb, b_emb, c_emb, a_emb, n_emb], axis=1)  # (B,5,k)
        sum_emb     = cp.sum(stacked, axis=1)                                  # (B,k)
        interaction = 0.5 * cp.sum(
            sum_emb ** 2 - cp.sum(stacked ** 2, axis=1), axis=1
        )
        return linear + interaction

    def sgd_step_batch(self, user_idx, biz_idx, city_idx, cat_idx, num_feat, ratings):
        batch_size = len(user_idx)

        # ===== FORWARD =====
        linear  = cp.full(batch_size, self.bias, dtype=cp.float32)
        linear += self.user_lin[user_idx]
        linear += self.biz_lin[biz_idx]
        linear += self.city_lin[city_idx]
        linear += self.cat_lin[cat_idx]
        linear += cp.sum(self.num_lin * num_feat, axis=1)

        u_emb = self.user_emb[user_idx]
        b_emb = self.biz_emb[biz_idx]
        c_emb = self.city_emb[city_idx]
        a_emb = self.cat_emb[cat_idx]
        n_emb = cp.dot(num_feat, self.num_emb)

        # Dropout — 1 mask duy nhất, dùng lại cho backward
        if self.training and self.dropout_rate > 0:
            inv = cp.float32(1.0 / (1.0 - self.dropout_rate))
            def make_mask():
                return (cp.random.rand(batch_size, self.k).astype(cp.float32) > self.dropout_rate) * inv
            u_mask = make_mask(); b_mask = make_mask()
            c_mask = make_mask(); a_mask = make_mask(); n_mask = make_mask()
            u_d = u_emb * u_mask; b_d = b_emb * b_mask
            c_d = c_emb * c_mask; a_d = a_emb * a_mask; n_d = n_emb * n_mask
        else:
            u_d = u_emb; b_d = b_emb; c_d = c_emb; a_d = a_emb; n_d = n_emb
            u_mask = b_mask = c_mask = a_mask = n_mask = None

        stacked     = cp.stack([u_d, b_d, c_d, a_d, n_d], axis=1)  # (B,5,k)
        sum_emb     = cp.sum(stacked, axis=1)                        # (B,k)
        interaction = 0.5 * cp.sum(
            sum_emb ** 2 - cp.sum(stacked ** 2, axis=1), axis=1
        )

        predictions = linear + interaction
        errors      = predictions - ratings
        loss        = cp.mean(errors ** 2)

        reg_loss = self.reg * (
            cp.sum(self.user_lin[user_idx]  ** 2) +
            cp.sum(self.biz_lin[biz_idx]    ** 2) +
            cp.sum(self.city_lin[city_idx]  ** 2) +
            cp.sum(self.cat_lin[cat_idx]    ** 2) +
            cp.sum(self.num_lin             ** 2) +
            cp.sum(self.user_emb[user_idx]  ** 2) +
            cp.sum(self.biz_emb[biz_idx]    ** 2) +
            cp.sum(self.city_emb[city_idx]  ** 2) +
            cp.sum(self.cat_emb[cat_idx]    ** 2) +
            cp.sum(self.num_emb             ** 2)
        ) / batch_size
        total_loss = loss + reg_loss

        # ===== BACKWARD =====
        e     = errors / batch_size   # (B,)
        e_col = e[:, None]            # (B,1)

        # Bias
        self.bias -= self.lr * cp.mean(errors)

        # Linear weights — scatter add
        def update_lin(param, idx, n_total):
            g = cp.zeros(n_total, dtype=cp.float32)
            cp.add.at(g, idx, e)
            g += self.reg * param
            param -= self.lr * self._clip(g)
            return param

        self.user_lin = update_lin(self.user_lin, user_idx, self.n_users)
        self.biz_lin  = update_lin(self.biz_lin,  biz_idx,  self.n_businesses)
        self.city_lin = update_lin(self.city_lin, city_idx, self.n_cities)
        self.cat_lin  = update_lin(self.cat_lin,  cat_idx,  self.n_cats)

        g_nl = cp.dot(num_feat.T, e) + self.reg * self.num_lin
        self.num_lin -= self.lr * self._clip(g_nl)

        # Embedding gradients
        def update_emb(param, emb_d, mask, idx, n_total):
            g = e_col * (sum_emb - emb_d)
            if mask is not None:
                g = g * mask
            g_acc = cp.zeros((n_total, self.k), dtype=cp.float32)
            cp.add.at(g_acc, idx, g)
            g_acc += self.reg * param
            param -= self.lr * self._clip(g_acc)
            return param

        self.user_emb = update_emb(self.user_emb, u_d, u_mask, user_idx, self.n_users)
        self.biz_emb  = update_emb(self.biz_emb,  b_d, b_mask, biz_idx,  self.n_businesses)
        self.city_emb = update_emb(self.city_emb,  c_d, c_mask, city_idx, self.n_cities)
        self.cat_emb  = update_emb(self.cat_emb,   a_d, a_mask, cat_idx,  self.n_cats)

        # Numerical embedding
        g_n   = e_col * (sum_emb - n_d)
        if n_mask is not None:
            g_n = g_n * n_mask
        g_n_acc = cp.dot(num_feat.T, g_n) + self.reg * self.num_emb
        self.num_emb -= self.lr * self._clip(g_n_acc)

        return float(total_loss)


# =========================================================
# METRICS
# =========================================================
def compute_metrics(predictions, targets):
    predictions = cp.asnumpy(predictions)
    targets     = cp.asnumpy(targets)
    mse    = np.mean((predictions - targets) ** 2)
    rmse   = np.sqrt(mse)
    mae    = np.mean(np.abs(predictions - targets))
    ss_res = np.sum((targets - predictions) ** 2)
    ss_tot = np.sum((targets - np.mean(targets)) ** 2)
    r2     = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}


def print_metrics(metrics, prefix=""):
    print(f"{prefix}MSE: {metrics['MSE']:.4f} | RMSE: {metrics['RMSE']:.4f} | "
          f"MAE: {metrics['MAE']:.4f} | R2: {metrics['R2']:.4f}")


def get_lr_decay(epoch, base_lr, decay_rate=0.99):
    return base_lr * (decay_rate ** epoch)


def predict_in_chunks(model, user_idx, biz_idx, city_idx, cat_idx,
                      num_feat, chunk_size=50000):
    preds = []
    for i in range(0, len(user_idx), chunk_size):
        s, e = i, min(i + chunk_size, len(user_idx))
        p = model.predict_batch(
            user_idx[s:e], biz_idx[s:e],
            city_idx[s:e], cat_idx[s:e], num_feat[s:e]
        )
        preds.append(p)
    return cp.concatenate(preds)


# =========================================================
# TRAINING
# =========================================================
def train_fm(user_indices, biz_indices, city_indices, cat_indices,
             num_features, ratings,
             n_users, n_businesses, n_cities, n_cats, num_dim,
             epochs=200, batch_size=8192, val_split=0.1,
             base_lr=0.01, k=128, reg=0.001, dropout_rate=0.3,
             decay_rate=0.99):

    print("\n" + "="*70)
    print("🚀 TRAINING FM — City+Category Embeddings (k=128, 7M data)")
    print("="*70)

    n_samples = len(user_indices)
    indices   = np.arange(n_samples)
    np.random.shuffle(indices)
    val_size  = int(n_samples * val_split)
    val_idx   = indices[:val_size]
    train_idx = indices[val_size:]

    print(f"\nDataset split:")
    print(f"  Train: {len(train_idx):,} | Val: {len(val_idx):,}")

    print("\nMoving data to GPU...")
    user_gpu = cp.array(user_indices)
    biz_gpu  = cp.array(biz_indices)
    city_gpu = cp.array(city_indices)
    cat_gpu  = cp.array(cat_indices)
    num_gpu  = cp.array(num_features)
    rat_gpu  = cp.array(ratings)
    print("  ✓ Data on GPU")

    baseline_mean = float(cp.mean(rat_gpu[train_idx]))
    baseline_rmse = float(cp.sqrt(cp.mean((rat_gpu[val_idx] - baseline_mean) ** 2)))
    print(f"\n📊 Baseline RMSE: {baseline_rmse:.4f}")

    model = FactorizationMachine(
        n_users=n_users, n_businesses=n_businesses,
        n_cities=n_cities, n_cats=n_cats, num_dim=num_dim,
        k=k, learning_rate=base_lr, reg=reg, dropout_rate=dropout_rate
    )

    history          = {'train_loss': [], 'val_rmse': [], 'lr': []}
    best_val_rmse    = float('inf')
    best_model_state = None
    patience         = 20
    patience_counter = 0

    print(f"\nConfig: Epochs={epochs} | Batch={batch_size} | LR={base_lr} (decay={decay_rate})")
    print(f"        k={k} | Reg={reg} | Dropout={dropout_rate} | Patience={patience}")
    print("\n" + "-"*70)

    for epoch in range(epochs):
        epoch_start = time.time()

        current_lr = get_lr_decay(epoch, base_lr, decay_rate)
        model.lr   = current_lr
        history['lr'].append(current_lr)

        model.training = True
        np.random.shuffle(train_idx)

        total_loss = 0.0
        n_batches  = (len(train_idx) + batch_size - 1) // batch_size

        for batch_num in range(n_batches):
            s   = batch_num * batch_size
            e   = min(s + batch_size, len(train_idx))
            idx = train_idx[s:e]

            batch_loss = model.sgd_step_batch(
                user_gpu[idx], biz_gpu[idx],
                city_gpu[idx], cat_gpu[idx],
                num_gpu[idx],  rat_gpu[idx]
            )
            total_loss += batch_loss * (e - s)

            if math.isnan(batch_loss) or batch_loss > 1000:
                print(f"\n⚠️  Loss exploded! Stopping.")
                return model, history

        avg_loss = total_loss / len(train_idx)
        history['train_loss'].append(avg_loss)

        # Validation
        model.training = False
        val_preds   = model.predict_batch(
            user_gpu[val_idx], biz_gpu[val_idx],
            city_gpu[val_idx], cat_gpu[val_idx], num_gpu[val_idx]
        )
        val_metrics = compute_metrics(val_preds, rat_gpu[val_idx])
        history['val_rmse'].append(val_metrics['RMSE'])

        epoch_time   = time.time() - epoch_start
        beating_base = "✓" if val_metrics['RMSE'] < baseline_rmse else "✗"

        print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {avg_loss:7.4f} | "
              f"Val RMSE: {val_metrics['RMSE']:.4f} {beating_base} | "
              f"MAE: {val_metrics['MAE']:.4f} | R2: {val_metrics['R2']:6.4f} | "
              f"LR: {current_lr:.6f} | {epoch_time:.1f}s")

        if val_metrics['RMSE'] < best_val_rmse:
            best_val_rmse    = val_metrics['RMSE']
            patience_counter = 0
            best_model_state = {
                'bias':     cp.asnumpy(model.bias),
                'user_lin': cp.asnumpy(model.user_lin),
                'biz_lin':  cp.asnumpy(model.biz_lin),
                'city_lin': cp.asnumpy(model.city_lin),
                'cat_lin':  cp.asnumpy(model.cat_lin),
                'num_lin':  cp.asnumpy(model.num_lin),
                'user_emb': cp.asnumpy(model.user_emb),
                'biz_emb':  cp.asnumpy(model.biz_emb),
                'city_emb': cp.asnumpy(model.city_emb),
                'cat_emb':  cp.asnumpy(model.cat_emb),
                'num_emb':  cp.asnumpy(model.num_emb),
            }
            print(f"  🎯 New best! RMSE: {best_val_rmse:.4f} "
                  f"(Δ: {baseline_rmse - best_val_rmse:+.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n⏹  Early stopping at epoch {epoch+1}")
                if best_model_state:
                    model.bias     = cp.array(best_model_state['bias'])
                    model.user_lin = cp.array(best_model_state['user_lin'])
                    model.biz_lin  = cp.array(best_model_state['biz_lin'])
                    model.city_lin = cp.array(best_model_state['city_lin'])
                    model.cat_lin  = cp.array(best_model_state['cat_lin'])
                    model.num_lin  = cp.array(best_model_state['num_lin'])
                    model.user_emb = cp.array(best_model_state['user_emb'])
                    model.biz_emb  = cp.array(best_model_state['biz_emb'])
                    model.city_emb = cp.array(best_model_state['city_emb'])
                    model.cat_emb  = cp.array(best_model_state['cat_emb'])
                    model.num_emb  = cp.array(best_model_state['num_emb'])
                    print("   ✓ Restored best model")
                break

    # Final evaluation
    print("\n" + "="*70)
    print("FINAL EVALUATION")
    print("="*70)
    model.training = False

    train_preds   = predict_in_chunks(model,
        user_gpu[train_idx], biz_gpu[train_idx],
        city_gpu[train_idx], cat_gpu[train_idx], num_gpu[train_idx])
    train_metrics = compute_metrics(train_preds, rat_gpu[train_idx])

    val_preds   = predict_in_chunks(model,
        user_gpu[val_idx], biz_gpu[val_idx],
        city_gpu[val_idx], cat_gpu[val_idx], num_gpu[val_idx])
    val_metrics = compute_metrics(val_preds, rat_gpu[val_idx])

    print(f"\n📊 Baseline RMSE: {baseline_rmse:.4f}")
    print("\n📈 Training Set:");  print_metrics(train_metrics, "  ")
    print("\n📈 Validation Set:"); print_metrics(val_metrics, "  ")

    improvement = baseline_rmse - val_metrics['RMSE']
    print(f"\n{'✓' if improvement > 0 else '✗'} Improvement: {improvement:+.4f}")

    overfit = train_metrics['RMSE'] - val_metrics['RMSE']
    print(f"📉 Overfitting gap: {overfit:.4f}")
    if abs(overfit) < 0.3:
        print("   ✓ Good generalization!")
    elif abs(overfit) < 0.5:
        print("   ⚠️  Moderate overfitting")
    else:
        print("   ✗ High overfitting")

    return model, history


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":

    BASE = "/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset"

    print("="*70)
    print("🚀 FM — City+Category Embeddings | k=128 | Full 7M | 200 epochs")
    print("="*70 + "\n")

    user_ctx                  = load_user_context(f"{BASE}/yelp_academic_dataset_user.json")
    biz_ctx, n_cities, n_cats = load_business_context(f"{BASE}/yelp_academic_dataset_business.json")
    checkin_ctx               = load_checkin_context(f"{BASE}/yelp_academic_dataset_checkin.json")
    user_tip, biz_tip         = load_tip_context(f"{BASE}/yelp_academic_dataset_tip.json")

    user_map = IDMapper()
    biz_map  = IDMapper()

    (user_indices, biz_indices, city_indices, cat_indices,
     num_features, ratings) = load_reviews(
        review_path=f"{BASE}/yelp_academic_dataset_review.json",
        user_ctx=user_ctx,
        biz_ctx=biz_ctx,
        checkin_ctx=checkin_ctx,
        user_tip=user_tip,
        biz_tip=biz_tip,
        user_map=user_map,
        biz_map=biz_map,
        max_reviews=None   # full 7M
    )

    model, history = train_fm(
        user_indices=user_indices,
        biz_indices=biz_indices,
        city_indices=city_indices,
        cat_indices=cat_indices,
        num_features=num_features,
        ratings=ratings,
        n_users=user_map.size(),
        n_businesses=biz_map.size(),
        n_cities=n_cities,
        n_cats=n_cats,
        num_dim=num_features.shape[1],
        epochs=200,
        batch_size=8192,
        val_split=0.1,
        base_lr=0.01,
        k=128,
        reg=0.001,
        dropout_rate=0.3,
        decay_rate=0.99,
    )

    print("\n✓ Training completed!")

    import os
    os.makedirs('/kaggle/working/models', exist_ok=True)

    np.savez('/kaggle/working/models/fm_city_cat_emb_k128.npz',
        bias=cp.asnumpy(model.bias),
        user_lin=cp.asnumpy(model.user_lin),
        biz_lin=cp.asnumpy(model.biz_lin),
        city_lin=cp.asnumpy(model.city_lin),
        cat_lin=cp.asnumpy(model.cat_lin),
        num_lin=cp.asnumpy(model.num_lin),
        user_emb=cp.asnumpy(model.user_emb),
        biz_emb=cp.asnumpy(model.biz_emb),
        city_emb=cp.asnumpy(model.city_emb),
        cat_emb=cp.asnumpy(model.cat_emb),
        num_emb=cp.asnumpy(model.num_emb)
    )
    print("  ✓ Model saved to /kaggle/working/models/fm_city_cat_emb_k128.npz")

    history_serializable = {
        k: [float(v) for v in vals]
        for k, vals in history.items()
    }
    with open('/kaggle/working/models/training_history_city_cat_emb.json', 'w') as f:
        json.dump(history_serializable, f, indent=2)
    print("  ✓ Training history saved")

🚀 FM — City+Category Embeddings | k=128 | Full 7M | 200 epochs

Loading user context from /kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_user.json...
  ✓ Loaded 1,987,897 users
Loading business context from /kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_business.json...
  ✓ Loaded 150,346 businesses
  ✓ Unique cities: 1,381, categories: 1,160
Loading checkin context from /kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_checkin.json...
  ✓ Loaded 131,930 businesses with checkin data
Loading tip context from /kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_tip.json...
  ✓ Loaded tips for 301,758 users and 106,193 businesses

Loading reviews from /kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_review.json...
  Loaded 100,000 reviews...
  Loaded 200,000 reviews...
  Loaded 300,000 reviews...
  Loaded 400,000